[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter9/graphrag-query.ipynb)

# Chapter 9: GraphRAG Query Notebook

This notebook demonstrates querying the knowledge graph built from movie scripts using GraphRAG.

GraphRAG provides two main query modes:
- **Global Search**: Answers questions about the entire dataset using community summaries
- **Local Search**: Answers questions about specific entities using the knowledge graph

**What you'll learn:**
- Load and explore GraphRAG's indexed output (entities, communities, relationships)
- Run global search queries for high-level thematic analysis
- Run local search queries for entity-specific information
- Compare global vs local search for different question types

**Prerequisites:** Run `graphrag-create.ipynb` first to build the knowledge graph. Set `OPENAI_API_KEY`.

## Setup and Configuration

We point to the GraphRAG workspace, verify that the required parquet output files exist, and confirm everything is ready for querying.

In [1]:
import os
import pandas as pd
from pathlib import Path

# Configuration
GRAPHRAG_DIR = Path("./graphrag_workspace")
OUTPUT_DIR = GRAPHRAG_DIR / "output"

# Verify GraphRAG output exists
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"GraphRAG output directory not found: {OUTPUT_DIR}\n"
        "Please run graphrag-create.ipynb first to build the knowledge graph."
    )

# Check for required files
required_files = [
    "entities.parquet",
    "relationships.parquet",
    "communities.parquet",
    "community_reports.parquet",
    "text_units.parquet"
]

missing_files = [f for f in required_files if not (OUTPUT_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing required GraphRAG output files: {missing_files}\n"
        "Please run graphrag-create.ipynb first."
    )

print("✅ GraphRAG output files found")
print(f"\nOutput directory: {OUTPUT_DIR}")
for file in required_files:
    size_kb = (OUTPUT_DIR / file).stat().st_size / 1024
    print(f"  - {file}: {size_kb:.1f} KB")

✅ GraphRAG output files found

Output directory: graphrag_workspace/output
  - entities.parquet: 1758.2 KB
  - relationships.parquet: 2080.7 KB
  - communities.parquet: 964.7 KB
  - community_reports.parquet: 9822.6 KB
  - text_units.parquet: 2700.5 KB


## Load GraphRAG Data

We read all five parquet tables produced by the indexing pipeline — entities, relationships, communities, community reports, and text units — into DataFrames for search.

In [2]:
# Load configuration and data
from graphrag.config.load_config import load_config

# Load configuration
config = load_config(root_dir=GRAPHRAG_DIR)

# Load parquet files
entities_df = pd.read_parquet(OUTPUT_DIR / "entities.parquet")
relationships_df = pd.read_parquet(OUTPUT_DIR / "relationships.parquet")
communities_df = pd.read_parquet(OUTPUT_DIR / "communities.parquet")
reports_df = pd.read_parquet(OUTPUT_DIR / "community_reports.parquet")
text_units_df = pd.read_parquet(OUTPUT_DIR / "text_units.parquet")

print(f"Loaded GraphRAG data:")
print(f"  - Entities: {len(entities_df):,}")
print(f"  - Relationships: {len(relationships_df):,}")
print(f"  - Communities: {len(communities_df):,}")
print(f"  - Community Reports: {len(reports_df):,}")
print(f"  - Text Units: {len(text_units_df):,}")

Model config based on fnllm is deprecated and will be removed in GraphRAG v3, please use ModelType.Chat or ModelType.Embedding instead to switch to LiteLLM config.
Model config based on fnllm is deprecated and will be removed in GraphRAG v3, please use ModelType.Chat or ModelType.Embedding instead to switch to LiteLLM config.


Loaded GraphRAG data:
  - Entities: 7,245
  - Relationships: 13,865
  - Communities: 1,292
  - Community Reports: 1,292
  - Text Units: 995


## Quick Data Exploration

We inspect the entity type distribution and community hierarchy to verify the indexing pipeline produced a well-structured knowledge graph before running queries.

In [3]:
# Entity type distribution
print("Entity Types:")
print(entities_df['type'].value_counts())

print("\n" + "="*60)
print("Sample Entities:")
print(entities_df[['title', 'type', 'description']].head(10))

Entity Types:
type
OBJECT          2663
LOCATION        1616
CHARACTER       1068
PERSON           729
EVENT            402
                 395
ORGANIZATION     372
Name: count, dtype: int64

Sample Entities:
                   title       type  \
0   "THE MALTESE FALCON"     OBJECT   
1       DASHIELL HAMMETT     PERSON   
2            JOHN HUSTON     PERSON   
3           HENRY BLANKE     PERSON   
4          ARTHUR EDESON     PERSON   
5           SAMUEL SPADE  CHARACTER   
6           EFFIE PERINE  CHARACTER   
7   BRIGID O'SHAUGHNESSY  CHARACTER   
8           MILES ARCHER  CHARACTER   
9  DETECTIVE TOM POLHAUS  CHARACTER   

                                         description  
0  The Maltese Falcon is the title object and cen...  
1  Dashiell Hammett is the author of the original...  
2  John Huston is the screenwriter and director o...  
3  Henry Blanke is the producer of the final vers...  
4  Arthur Edeson is the cinematographer (photogra...  
5  Samuel Spade is a private i

In [4]:
# Community distribution by level
print("Community Hierarchy:")
print(communities_df['level'].value_counts().sort_index())

print("\n" + "="*60)
print("Sample Community Report:")
if len(reports_df) > 0:
    print(f"\nTitle: {reports_df.iloc[0]['title']}")
    print(f"\nSummary:\n{reports_df.iloc[0]['summary']}")

Community Hierarchy:
level
0     22
1    273
2    699
3    271
4     27
Name: count, dtype: int64

Sample Community Report:

Title: Jonah Baldwin and Family Dynamics

Summary:
This community centers on Jonah Baldwin, a nine-year-old boy deeply connected to his father, Sam Baldwin, and affected by the loss of his mother, Margaret Abbott Baldwin. The entities and relationships depict Jonah's daily life, emotional experiences, and social interactions, including his close bond with Sam, his involvement in family and social events, and his navigation of grief and growing up. Key objects and locations such as the dingy, fishing rod, houseboat, and various personal items illustrate the environment and activities that shape Jonah's life. The community also includes social connections like friends and media interactions, highlighting Jonah's broader social context.


## Initialize Search Engines

GraphRAG provides two search modes:
1. **Global Search**: Uses community summaries to answer broad questions
2. **Local Search**: Uses entity relationships for specific questions

In [5]:
# We'll use the simpler GraphRAG API that works directly with DataFrames
# No need to convert to data model objects
from graphrag.api.query import global_search, local_search

print("✅ Ready to query using GraphRAG API")
print("\nLoaded DataFrames:")
print(f"  - Entities: {len(entities_df):,}")
print(f"  - Communities: {len(communities_df):,}")
print(f"  - Reports: {len(reports_df):,}")
print(f"  - Text Units: {len(text_units_df):,}")
print(f"  - Relationships: {len(relationships_df):,}")

✅ Ready to query using GraphRAG API

Loaded DataFrames:
  - Entities: 7,245
  - Communities: 1,292
  - Reports: 1,292
  - Text Units: 995
  - Relationships: 13,865


In [6]:
# Helper function for global search
async def run_global_search(query: str, verbose: bool = False):
    """Run a global search query."""
    result, context = await global_search(
        config=config,
        entities=entities_df,
        communities=communities_df,
        community_reports=reports_df,
        community_level=2,  # Use level 2 communities for good balance
        dynamic_community_selection=False,
        response_type="multiple paragraphs",
        query=query,
        verbose=verbose
    )
    return result, context

print("✅ Global search helper function ready")

✅ Global search helper function ready


In [7]:
# Helper function for local search
async def run_local_search(query: str, verbose: bool = False):
    """Run a local search query."""
    result, context = await local_search(
        config=config,
        entities=entities_df,
        communities=communities_df,
        community_reports=reports_df,
        text_units=text_units_df,
        relationships=relationships_df,
        covariates=None,  # No covariates in this example
        community_level=2,
        response_type="multiple paragraphs",
        query=query,
        verbose=verbose
    )
    return result, context

print("✅ Local search helper function ready")

✅ Local search helper function ready


## Global Search Examples

Global search is best for:
- High-level questions about themes and patterns
- Questions requiring synthesis across the entire dataset
- Comparative analysis
- Understanding overall structure and relationships

In [8]:
# Global Search Query 1: Themes
query = "What are the main themes and narrative patterns across all the movie scripts?"

print("="*80)
print(f"GLOBAL SEARCH QUERY: {query}")
print("="*80)

result, context = await run_global_search(query)
print(f"\nAnswer:\n{result}\n")

GLOBAL SEARCH QUERY: What are the main themes and narrative patterns across all the movie scripts?

Answer:
The movie scripts analyzed reveal a rich tapestry of recurring themes and narrative patterns that span a wide range of genres, character types, and settings. These themes often intertwine, creating complex stories that explore human emotions, supernatural elements, social dynamics, and conflict. Below is a comprehensive synthesis of the main themes and narrative patterns identified across the dataset.

---

## 1. **Conflict and Struggle**

Conflict is a pervasive and driving force in many narratives, manifesting in various forms:

- **Physical and Military Conflict:** Stories often depict intense battles and warfare, such as Conan’s confrontations with hostile tribes and empires, Vlad’s defense against Mehmed’s army, and the violent raids by the Turanian Horsemen. These conflicts emphasize themes of survival, leadership, rebellion, and the cost of violence [Data: Reports (177, 66

In [9]:
# Global Search Query 2: Character archetypes
query = "What types of characters and character archetypes appear most frequently across the movies?"

print("="*80)
print(f"GLOBAL SEARCH QUERY: {query}")
print("="*80)

result, context = await run_global_search(query)
print(f"\nAnswer:\n{result}\n")

GLOBAL SEARCH QUERY: What types of characters and character archetypes appear most frequently across the movies?

Answer:
Across the movies analyzed, several character types and archetypes appear most frequently, reflecting a rich diversity of narrative roles and thematic elements. These archetypes span heroic figures, investigators, family members, supernatural entities, and leaders, each contributing uniquely to their respective story communities.

### Heroic and Warrior Archetypes

One of the most prominent archetypes is the **heroic warrior or fighter**, exemplified by characters like Conan, the Cimmerian warrior. Conan embodies resilience, combat prowess, leadership, and cultural symbolism, often engaging in survival struggles, battles against oppressive forces, and quests for vengeance and justice. This archetype is multifaceted, showing both physical strength and moments of vulnerability and loyalty, serving as a central figure in action and adventure narratives [Data: Reports (

In [10]:
# Global Search Query 3: Conflicts and relationships
query = "What are the most common types of conflicts and relationships portrayed in these movies?"

print("="*80)
print(f"GLOBAL SEARCH QUERY: {query}")
print("="*80)

result, context = await run_global_search(query)
print(f"\nAnswer:\n{result}\n")

GLOBAL SEARCH QUERY: What are the most common types of conflicts and relationships portrayed in these movies?

Answer:
The movies portray a rich and diverse array of conflicts and relationships, reflecting complex human experiences across various settings and genres. The most common types of conflicts and relationships can be broadly categorized into physical, supernatural, familial, investigative, political, and interpersonal struggles, each with distinctive themes and narrative implications.

### Physical and Violent Conflicts

One of the most prevalent conflict types involves intense physical combat and violent confrontations. These include battles between warriors, such as Conan’s numerous fights against hostile groups like the Picts and Khalar Singh’s forces, which often emphasize survival, vengeance, and resistance against oppression [Data: Reports (177, 8, 170, 6, 699)]. Similarly, boxing rivalries, notably between Rocky Balboa and opponents like Clubber Lang and Apollo Creed, h

In [11]:
# Global Search Query 4: Settings and locations
query = "What are the most common settings and locations used in these movie scripts?"

print("="*80)
print(f"GLOBAL SEARCH QUERY: {query}")
print("="*80)

result, context = await run_global_search(query)
print(f"\nAnswer:\n{result}\n")

GLOBAL SEARCH QUERY: What are the most common settings and locations used in these movie scripts?

Answer:
The dataset reveals a rich variety of settings and locations commonly used across multiple movie scripts, reflecting diverse narrative needs such as domestic life, supernatural events, urban environments, and specialized community hubs. These settings serve as critical backdrops that shape the tone, character interactions, and plot developments in their respective stories.

### Domestic and Family Homes

One of the most prevalent types of settings are domestic environments, particularly family homes. These include detailed multi-room houses such as Lorraine's House, the Lambert family household, the Rose House, and various other family residences. These homes often serve as the primary physical and narrative loci where much of the interpersonal drama, supernatural occurrences, and emotional turmoil unfold. For example, Lorraine's House is a central setting for supernatural events 

## Local Search Examples

Local search is best for:
- Questions about specific entities (characters, locations, events)
- Detailed information about relationships
- Finding specific details in the text
- Entity-centric queries

In [12]:
# First, let's see what main characters we have
print("Top 20 most frequently mentioned entities:")
top_entities = entities_df.nlargest(20, 'degree')[['title', 'type', 'degree']]
print(top_entities.to_string())

Top 20 most frequently mentioned entities:
         title       type  degree
3039    WELLES     PERSON     440
115        TOM     PERSON     283
1671  JONATHAN  CHARACTER     240
47       SPADE  CHARACTER     225
4421      VLAD                224
5619     MIKEY     PERSON     221
2130       RAY  CHARACTER     216
102        SAM                212
2131      CARL  CHARACTER     204
4825   MALCOLM     PERSON     194
1654   BARBARA     PERSON     189
6735   BRENDAN  CHARACTER     177
3708     KAREN  CHARACTER     175
1235      JANE  CHARACTER     173
5346     ALIKE     PERSON     170
4035     CONAN  CHARACTER     168
705      ROCKY  CHARACTER     167
6699     MARSH  CHARACTER     165
2689     ANNIE     PERSON     142
3205      MARY  CHARACTER     127


In [13]:
# Local Search Query 1: Specific character
# Note: Replace with an actual character name from your dataset
query = "Tell me about the character Tom Welles and their role in the story"

print("="*80)
print(f"LOCAL SEARCH QUERY: {query}")
print("="*80)

result, context = await run_local_search(query)
print(f"\nAnswer:\n{result}\n")

LOCAL SEARCH QUERY: Tell me about the character Tom Welles and their role in the story

Answer:
### Overview of Tom Welles

Tom Welles is a licensed private investigator based in the Commonwealth of Pennsylvania. He specializes in complex and sensitive cases, including missing persons, corporate investigations, and underground criminal activities. Tom Welles is a central figure in a multifaceted investigation involving the disappearance of Mary Anne Mathews, a missing girl believed to be a runaway, and a possible murder connected to illicit films and financial irregularities.

### Role in the Story

Welles operates independently but collaborates with various organizations such as the Royal Canadian Mounted Police (R.C.M.P.) and the U.S. Resource Center for Missing Persons. His work is methodical and cautious, involving detailed archival research, surveillance, and forensic analysis. He uses photographic enlargers, 8mm projectors, and archival research in locations like Cleveland and Qu

In [14]:
# Local Search Query 2: Relationships
query = "What relationships and interactions exist between the main characters?"

print("="*80)
print(f"LOCAL SEARCH QUERY: {query}")
print("="*80)

result, context = await run_local_search(query)
print(f"\nAnswer:\n{result}\n")

LOCAL SEARCH QUERY: What relationships and interactions exist between the main characters?

Answer:
The main characters exhibit a complex web of relationships and interactions that are deeply intertwined with family dynamics, personal struggles, and social connections.

### Barbara and Jonathan
Barbara and Jonathan share a multifaceted and turbulent marital relationship. They are depicted as a married couple with children, Eve and Josh, navigating the challenges of a troubled marriage and impending divorce. Their interactions range from intimate and caring moments—such as sharing dinners and managing family life—to confrontations marked by emotional conflict and physical altercations. Barbara is portrayed as a strong, resourceful woman managing both family and her catering business, while Jonathan is a lawyer balancing professional pressures with personal vulnerabilities. Despite their conflicts, there are moments of affection and mutual concern, especially regarding their children and

In [15]:
# Local Search Query 3: Specific event or location
query = "What important events take place in New York or similar urban settings?"

print("="*80)
print(f"LOCAL SEARCH QUERY: {query}")
print("="*80)

result, context = await run_local_search(query)
print(f"\nAnswer:\n{result}\n")

LOCAL SEARCH QUERY: What important events take place in New York or similar urban settings?

Answer:
# Important Events in New York and Similar Urban Settings

New York City, along with its boroughs such as Manhattan and Brooklyn, serves as a significant backdrop for various key events and narrative developments. The city’s iconic landmarks and urban environment provide a rich setting for personal, cultural, and social interactions.

## Planned Meeting on Valentine's Day at the Empire State Building

One of the most notable events is the planned meeting between Sam Baldwin, Jonah Baldwin, and Annie Reed on Valentine's Day. This meeting is set to take place on the observation deck of the Empire State Building, a landmark that symbolizes romance and hope. The building’s spectacular nighttime illumination, often featuring a pink and white heart, enhances the emotional significance of this rendezvous. This event is central to the narrative, representing a moment of reunion and connection a